# TinyVAD Project Analysis

This notebook provides a comprehensive analysis of the TinyVAD project, covering:

1. **Project Overview** — motivation, pipeline, and model architecture
2. **Knowledge Distillation** — training curves and compression efficiency
3. **TORGO Fine-tuning** — three strategies compared across 8 dysarthric speakers
4. **Catastrophic Forgetting** — quantifying and mitigating retention loss
5. **Final Conclusions** — key takeaways and trade-off summary

All results are sourced directly from experimental outputs in `distill_record.ipynb` and `torgo_finetune_record.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded.')

---
## 1. Project Overview

### Motivation
Voice Activity Detection (VAD) is a fundamental component in speech pipelines, but state-of-the-art models like SpeechBrain's CRDNN are too large for edge deployment. This project targets two goals:

1. **Compression** — distil CRDNN (109,700 params) into TinyVAD (~10K params)
2. **Personalisation** — fine-tune TinyVAD on dysarthric speech (TORGO dataset) without losing general VAD ability

### Pipeline
```
 LibriParty corpus                   TORGO corpus
      │                                    │
      ▼                                    ▼
 CRDNN teacher ──── KD loss ────► TinyVAD ──── LOSO fine-tuning ──► personalised VAD
 (109 700 params)   (soft + hard)  (9 937 params)   (3 strategies)
```

### Datasets
| Dataset | Role | Size |
|---------|------|------|
| LibriParty | Distillation train/val | 11 180 train / 1 932 val |
| TORGO | Fine-tuning + test | 8 dysarthric speakers (F01, F03, F04, M01, M02, M04, M05, MC01) |

---
## 2. Model Architecture — Teacher vs. Student

In [ ]:
# ── Architecture comparison table ──────────────────────────────────────────
arch_data = {
    'Model':      ['CRDNN Teacher', 'TinyVAD Student'],
    'Parameters': [109_700,          9_937],
    'CNN':        ['Conv2d stack (larger)', 'Conv2d 1→8→16, MaxPool'],
    'RNN':        ['Bidirectional LSTM/GRU', 'Unidirectional GRU, h=16'],
    'DNN':        ['Multi-layer DNN', 'Linear 16→8→1'],
    'Input':      ['40-mel, 10 ms hop', '40-mel, 10 ms hop'],
}
df_arch = pd.DataFrame(arch_data).set_index('Model')
print('=== Architecture Summary ===')
print(df_arch.to_string())
print(f'\nCompression ratio: {109700 / 9937:.1f}x fewer parameters')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
models = ['CRDNN\nTeacher', 'TinyVAD\nStudent']
params = [109_700, 9_937]
colors = ['#4C72B0', '#DD8452']
bars = ax.bar(models, params, color=colors, width=0.45, zorder=3)
ax.set_ylabel('Number of Parameters')
ax.set_title('Model Size: Teacher vs. Student')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.4, zorder=0)
for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1500,
            f'{p:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.annotate('', xy=(1, 9937), xytext=(0, 109700),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
ax.text(0.5, 60000, f'11×\nsmaller', ha='center', color='gray', fontsize=10)
plt.tight_layout()
plt.savefig('fig_model_size.png', bbox_inches='tight')
plt.show()

---
## 3. Knowledge Distillation — Training Curves

TinyVAD is trained for 30 epochs using knowledge distillation from the CRDNN teacher on the LibriParty dataset.
The loss is a combination of:
- **Soft loss** — KL divergence between teacher and student output probabilities (temperature-scaled)
- **Hard loss** — binary cross-entropy against ground-truth VAD labels

In [ ]:
# Training log extracted from distill_record.ipynb
distill_log = [
    (1,  0.5606, 0.8655, 0.8003, 0.9423),
    (2,  0.5035, 0.9038, 0.8950, 0.9127),
    (3,  0.4889, 0.9050, 0.8903, 0.9201),
    (4,  0.4818, 0.9099, 0.8935, 0.9270),
    (5,  0.4782, 0.9133, 0.9107, 0.9159),
    (6,  0.4755, 0.9102, 0.8979, 0.9229),
    (7,  0.4733, 0.9122, 0.9052, 0.9193),
    (8,  0.4710, 0.9141, 0.9040, 0.9244),
    (9,  0.4701, 0.9143, 0.9038, 0.9251),
    (10, 0.4694, 0.9167, 0.9101, 0.9235),
    (11, 0.4682, 0.9122, 0.8881, 0.9377),
    (12, 0.4666, 0.9151, 0.8968, 0.9343),
    (13, 0.4657, 0.9113, 0.8899, 0.9339),
    (14, 0.4656, 0.9157, 0.8998, 0.9322),
    (15, 0.4638, 0.9166, 0.9073, 0.9262),
    (16, 0.4635, 0.9178, 0.9079, 0.9279),
    (17, 0.4632, 0.9193, 0.9113, 0.9275),
    (18, 0.4629, 0.9194, 0.9066, 0.9324),
    (19, 0.4625, 0.9180, 0.9101, 0.9259),
    (20, 0.4620, 0.9178, 0.9011, 0.9352),
    (21, 0.4620, 0.9200, 0.9184, 0.9216),
    (22, 0.4616, 0.9190, 0.9073, 0.9309),
    (23, 0.4614, 0.9175, 0.9002, 0.9355),
    (24, 0.4610, 0.9172, 0.8978, 0.9376),
    (25, 0.4606, 0.9196, 0.9057, 0.9339),
    (26, 0.4597, 0.9197, 0.9067, 0.9332),
    (27, 0.4596, 0.9204, 0.9082, 0.9331),
    (28, 0.4589, 0.9203, 0.9079, 0.9331),
    (29, 0.4590, 0.9177, 0.8970, 0.9394),
    (30, 0.4590, 0.9223, 0.9168, 0.9280),
]
epochs = [r[0] for r in distill_log]
losses = [r[1] for r in distill_log]
f1s    = [r[2] for r in distill_log]
precs  = [r[3] for r in distill_log]
recs   = [r[4] for r in distill_log]

# Best model epochs (marked in training log)
best_epochs = [1, 2, 3, 4, 5, 8, 9, 10, 16, 17, 18, 21, 27, 30]
print(f'Final (epoch 30) → Loss: {losses[-1]:.4f} | F1: {f1s[-1]:.4f} | Prec: {precs[-1]:.4f} | Rec: {recs[-1]:.4f}')
print(f'Best val F1 achieved: {max(f1s):.4f} at epoch {epochs[f1s.index(max(f1s))]}')
print(f'Teacher LibriParty F1: 0.9463  |  Student LibriParty F1: {max(f1s):.4f}')
print(f'Gap to teacher: {0.9463 - max(f1s):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Left: Loss curve ──────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs, losses, color='#C44E52', lw=2, marker='o', ms=3, label='Train Loss')
best_loss = [losses[e-1] for e in best_epochs]
ax.scatter(best_epochs, best_loss, s=60, color='gold', edgecolors='#C44E52',
           zorder=5, label='Best saved')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Distillation Loss over Training')
ax.set_xlim(0, 31)
ax.grid(alpha=0.3)
ax.legend()

# ── Right: F1 / Prec / Rec ────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs, f1s,   color='#2ca02c', lw=2, marker='o', ms=3, label='F1')
ax.plot(epochs, precs, color='#1f77b4', lw=2, marker='s', ms=3, label='Precision', alpha=0.8)
ax.plot(epochs, recs,  color='#ff7f0e', lw=2, marker='^', ms=3, label='Recall', alpha=0.8)
# teacher ceiling lines
ax.axhline(0.9463, color='#2ca02c', ls='--', lw=1.2, alpha=0.6, label='Teacher F1=0.9463')
ax.axhline(0.9446, color='#1f77b4', ls='--', lw=1.2, alpha=0.4)
ax.axhline(0.9481, color='#ff7f0e', ls='--', lw=1.2, alpha=0.4)
best_f1 = [f1s[e-1] for e in best_epochs]
ax.scatter(best_epochs, best_f1, s=60, color='gold', edgecolors='#2ca02c', zorder=5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Validation F1 / Precision / Recall')
ax.set_ylim(0.78, 0.98)
ax.set_xlim(0, 31)
ax.grid(alpha=0.3)
ax.legend(loc='lower right')

plt.suptitle('TinyVAD Knowledge Distillation — Training Progress (LibriParty val)', y=1.02)
plt.tight_layout()
plt.savefig('fig_distill_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Convergence analysis ───────────────────────────────────────────────────
f1_arr    = np.array(f1s)
loss_arr  = np.array(losses)

# Epoch at which F1 first exceeds 90%
above_90 = next(i+1 for i, v in enumerate(f1s) if v >= 0.90)
# F1 improvement in last 10 epochs vs first 10
early_gain = f1_arr[4] - f1_arr[0]   # epochs 1-5
late_gain  = f1_arr[-1] - f1_arr[19] # epochs 21-30
loss_drop  = losses[0] - losses[-1]

print('=== Distillation Convergence Analysis ===')
print(f'  F1 first exceeds 0.90 : epoch {above_90}')
print(f'  F1 gain (epochs 1→5)  : {early_gain:+.4f}  (fast initial learning)')
print(f'  F1 gain (epochs 21→30): {late_gain:+.4f}   (slow refinement)')
print(f'  Total loss reduction   : {loss_drop:.4f} ({losses[0]:.4f} → {losses[-1]:.4f})')
print(f'  Teacher F1 gap         : {0.9463 - max(f1s):.4f}  (student reaches {max(f1s):.4f})')
print()
print('Observation: most of the F1 improvement happens in the first 5 epochs.')
print('After epoch 10, gains are marginal (<0.006 F1), and loss plateaus around 0.460.')
print('The student closes most of the gap to the teacher (0.9223 vs 0.9463, gap=0.024).')

---
## 4. TORGO Fine-tuning — Experimental Setup

The TORGO dataset contains recordings from **dysarthric speakers** (motor speech disorder) and matched controls. We use 8 annotated speakers:

| Speaker | Type | Severity |
|---------|------|----------|
| F01 | Dysarthric female | — |
| F03 | Dysarthric female | — |
| F04 | Control female | — |
| M01 | Dysarthric male | Severe |
| M02 | Dysarthric male | Moderate |
| M04 | Dysarthric male | — |
| M05 | Dysarthric male | — |
| MC01 | Control male | — |

**LOSO (Leave-One-Speaker-Out)** protocol: each speaker is held out as the test set for their fold; the model is fine-tuned on all others and evaluated on the held-out speaker for TORGO F1, and on LibriParty val for retention (forgetting) measurement.

### Three strategies compared
| Strategy | Description |
|----------|-------------|
| **LOSO lr=1e-4** | Standard LOSO, default learning rate |
| **LOSO lr=5e-5** | Half the learning rate — slower, more conservative |
| **Mixed-data** | Replay LibriParty alongside TORGO during fine-tuning |

In [ ]:
speakers = ['F01', 'F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']

# Pre-trained baseline (same for all strategies)
pre_f1 = {'F01': 0.7911, 'F03': 0.7809, 'F04': 0.7645,
           'M01': 0.5943, 'M02': 0.7557, 'M04': 0.7182,
           'M05': 0.8145, 'MC01': 0.7069}

# ── Strategy 1: LOSO lr=1e-4 ──────────────────────────────────────────────
loso1e4 = {
    'torgo_f1':  {'F01':0.8724,'F03':0.8780,'F04':0.9063,'M01':0.6331,'M02':0.6620,'M04':0.7957,'M05':0.8464,'MC01':0.8421},
    'libri_f1':  {'F01':0.8563,'F03':0.8578,'F04':0.8550,'M01':0.8472,'M02':0.8526,'M04':0.8581,'M05':0.8627,'MC01':0.8743},
    'gain':      {'F01':0.0813,'F03':0.0971,'F04':0.1418,'M01':0.0388,'M02':-0.0937,'M04':0.0776,'M05':0.0318,'MC01':0.1351},
    'drop':      {'F01':0.0660,'F03':0.0646,'F04':0.0674,'M01':0.0751,'M02':0.0697,'M04':0.0643,'M05':0.0597,'MC01':0.0481},
}

# ── Strategy 2: LOSO lr=5e-5 ──────────────────────────────────────────────
loso5e5 = {
    'torgo_f1':  {'F01':0.8726,'F03':0.8722,'F04':0.8806,'M01':0.6067,'M02':0.7484,'M04':0.7331,'M05':0.8269,'MC01':0.8074},
    'libri_f1':  {'F01':0.8593,'F03':0.8604,'F04':0.8611,'M01':0.8678,'M02':0.9201,'M04':0.9023,'M05':0.9007,'MC01':0.8607},
    'gain':      {'F01':0.0815,'F03':0.0913,'F04':0.1162,'M01':0.0125,'M02':-0.0073,'M04':0.0149,'M05':0.0124,'MC01':0.1005},
    'drop':      {'F01':0.0630,'F03':0.0619,'F04':0.0612,'M01':0.0545,'M02':0.0022,'M04':0.0201,'M05':0.0217,'MC01':0.0616},
}

# ── Strategy 3: Mixed-data replay ─────────────────────────────────────────
mixed = {
    'torgo_f1':  {'F01':0.8594,'F03':0.8709,'F04':0.9065,'M01':0.6347,'M02':0.6958,'M04':0.7941,'M05':0.8469,'MC01':0.8406},
    'libri_f1':  {'F01':0.9171,'F03':0.9147,'F04':0.9094,'M01':0.9089,'M02':0.9149,'M04':0.9111,'M05':0.9093,'MC01':0.9085},
    'gain':      {'F01':0.0683,'F03':0.0900,'F04':0.1420,'M01':0.0404,'M02':-0.0599,'M04':0.0759,'M05':0.0324,'MC01':0.1337},
    'drop':      {'F01':0.0052,'F03':0.0076,'F04':0.0129,'M01':0.0134,'M02':0.0074,'M04':0.0113,'M05':0.0130,'MC01':0.0139},
}

# Reference
BASELINE_LIBRI = 0.9223
TEACHER_LIBRI  = 0.9463
TEACHER_TORGO  = 0.6320   # teacher on full TORGO
BASELINE_TORGO = 0.7368   # pre-trained on full TORGO

strategies = [
    ('LOSO lr=1e-4', loso1e4, '#4C72B0'),
    ('LOSO lr=5e-5', loso5e5, '#DD8452'),
    ('Mixed-data',   mixed,   '#55A868'),
]

print('Data loaded. Strategies:', [s[0] for s in strategies])

---
## 5. Per-Speaker TORGO F1 — Baseline vs. Fine-tuned

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

x = np.arange(len(speakers))
w = 0.35

for ax, (label, strat, color) in zip(axes, strategies):
    pre_vals  = [pre_f1[s]              for s in speakers]
    post_vals = [strat['torgo_f1'][s]   for s in speakers]
    gains     = [strat['gain'][s]       for s in speakers]

    bars_pre  = ax.bar(x - w/2, pre_vals,  w, label='Pre-trained', color='#aec6e8', edgecolor='white')
    bars_post = ax.bar(x + w/2, post_vals, w, label='Fine-tuned',  color=color,     edgecolor='white')

    # Annotate gain/loss
    for i, (bpre, bpost, g) in enumerate(zip(bars_pre, bars_post, gains)):
        clr  = '#2ca02c' if g >= 0 else '#d62728'
        sign = '+' if g >= 0 else ''
        ax.text(bpost.get_x() + bpost.get_width() / 2,
                bpost.get_height() + 0.005,
                f'{sign}{g:.2f}', ha='center', va='bottom',
                fontsize=7.5, color=clr, fontweight='bold')

    # Teacher on full TORGO (dashed)
    ax.axhline(TEACHER_TORGO, color='gray', ls=':', lw=1.4, label=f'Teacher TORGO={TEACHER_TORGO}')

    ax.set_xticks(x)
    ax.set_xticklabels(speakers, rotation=30, ha='right')
    ax.set_title(label)
    ax.set_ylim(0.40, 1.0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8.5, loc='lower right')

axes[0].set_ylabel('TORGO F1 (held-out speaker)')
plt.suptitle('Per-Speaker TORGO F1: Pre-trained vs. Fine-tuned (LOSO)', y=1.03)
plt.tight_layout()
plt.savefig('fig_per_speaker_torgo.png', bbox_inches='tight')
plt.show()

---
## 6. Catastrophic Forgetting — LibriParty Retention

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x   = np.arange(len(speakers))
n   = len(strategies)
w   = 0.22
off = [-w, 0, w]

for i, (label, strat, color) in enumerate(strategies):
    drops = [strat['drop'][s] for s in speakers]
    bars  = ax.bar(x + off[i], drops, w, label=label, color=color, edgecolor='white', alpha=0.88)

# 0.05 forgetting threshold
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='Forgetting threshold (0.05)')
ax.axhline(0.00, color='black', lw=0.6)

ax.set_xticks(x)
ax.set_xticklabels(speakers)
ax.set_ylabel('LibriParty F1 Drop (pre − post)')
ax.set_title('Catastrophic Forgetting per Speaker and Strategy')
ax.set_ylim(-0.01, 0.11)
ax.grid(axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('fig_forgetting.png', bbox_inches='tight')
plt.show()

In [ ]:
# Table: forgetting fraction above 0.05 threshold
threshold = 0.05
print(f'Speakers with LibriParty drop > {threshold} (significant forgetting):')
print(f'{"Speaker":<8} {"lr=1e-4":>9} {"lr=5e-5":>9} {"Mixed":>9}')
print('-' * 40)
forget_counts = [0, 0, 0]
for s in speakers:
    row = []
    for j, (_, strat, _) in enumerate(strategies):
        d = strat['drop'][s]
        flag = ' !' if d > threshold else '  '
        row.append(f'{d:.4f}{flag}')
        if d > threshold:
            forget_counts[j] += 1
    print(f'{s:<8}  {row[0]}  {row[1]}  {row[2]}')
print('-' * 40)

macro_drops = [[strat['drop'][s] for s in speakers] for _, strat, _ in strategies]
labels_row = ['lr=1e-4', 'lr=5e-5', 'Mixed']
print(f'{"Macro drop":<8}', '  '.join([f'{np.mean(d):>9.4f}  ' for d in macro_drops]))
print(f'{"# forgot":<8}', '  '.join([f'{c:>9}    ' for c in forget_counts]))

---
## 7. Learning Rate Sensitivity: lr=1e-4 vs. lr=5e-5

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(speakers))
w = 0.35

# ── TORGO F1 gain ─────────────────────────────────────────────────────────
ax = axes[0]
gains_1e4 = [loso1e4['gain'][s] for s in speakers]
gains_5e5 = [loso5e5['gain'][s] for s in speakers]
ax.bar(x - w/2, gains_1e4, w, label='lr=1e-4', color='#4C72B0', alpha=0.85)
ax.bar(x + w/2, gains_5e5, w, label='lr=5e-5', color='#DD8452', alpha=0.85)
ax.axhline(0, color='black', lw=0.7)
ax.set_xticks(x); ax.set_xticklabels(speakers, rotation=30, ha='right')
ax.set_ylabel('TORGO F1 Gain')
ax.set_title('TORGO Gain by Learning Rate')
ax.grid(axis='y', alpha=0.3)
ax.legend()

# ── LibriParty drop ───────────────────────────────────────────────────────
ax = axes[1]
drops_1e4 = [loso1e4['drop'][s] for s in speakers]
drops_5e5 = [loso5e5['drop'][s] for s in speakers]
ax.bar(x - w/2, drops_1e4, w, label='lr=1e-4', color='#4C72B0', alpha=0.85)
ax.bar(x + w/2, drops_5e5, w, label='lr=5e-5', color='#DD8452', alpha=0.85)
ax.axhline(0.05, color='red', ls='--', lw=1.2, label='Forgetting threshold')
ax.set_xticks(x); ax.set_xticklabels(speakers, rotation=30, ha='right')
ax.set_ylabel('LibriParty F1 Drop')
ax.set_title('Forgetting by Learning Rate')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.suptitle('Effect of Learning Rate on Adaptation–Forgetting Trade-off', y=1.02)
plt.tight_layout()
plt.savefig('fig_lr_comparison.png', bbox_inches='tight')
plt.show()

# Summary
print(f'\nMacro TORGO gain  — lr=1e-4: {np.mean(gains_1e4):+.4f} | lr=5e-5: {np.mean(gains_5e5):+.4f}')
print(f'Macro Libri drop  — lr=1e-4: {np.mean(drops_1e4):+.4f} | lr=5e-5: {np.mean(drops_5e5):+.4f}')
print()
print('Key finding: lr=5e-5 significantly reduces forgetting for M02 and M04')
print('but also reduces adaptation gain (M01 gain drops from +0.039 to +0.013).')
print('Trade-off: lower LR is safer but less adaptive for hard speakers like M01.')

---
## 8. Mixed-data Replay — Anti-Forgetting Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(speakers))
w = 0.25

# ── TORGO F1 gain ─────────────────────────────────────────────────────────
ax = axes[0]
gains_mix = [mixed['gain'][s] for s in speakers]
ax.bar(x - w,   gains_1e4, w, label='LOSO lr=1e-4', color='#4C72B0', alpha=0.85)
ax.bar(x,       gains_5e5, w, label='LOSO lr=5e-5', color='#DD8452', alpha=0.85)
ax.bar(x + w,   gains_mix, w, label='Mixed-data',   color='#55A868', alpha=0.85)
ax.axhline(0, color='black', lw=0.7)
ax.set_xticks(x); ax.set_xticklabels(speakers, rotation=30, ha='right')
ax.set_ylabel('TORGO F1 Gain')
ax.set_title('TORGO Adaptation Gain')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)

# ── LibriParty drop ───────────────────────────────────────────────────────
ax = axes[1]
drops_mix = [mixed['drop'][s] for s in speakers]
ax.bar(x - w,   drops_1e4, w, label='LOSO lr=1e-4', color='#4C72B0', alpha=0.85)
ax.bar(x,       drops_5e5, w, label='LOSO lr=5e-5', color='#DD8452', alpha=0.85)
ax.bar(x + w,   drops_mix, w, label='Mixed-data',   color='#55A868', alpha=0.85)
ax.axhline(0.05, color='red', ls='--', lw=1.2, label='Threshold=0.05')
ax.set_xticks(x); ax.set_xticklabels(speakers, rotation=30, ha='right')
ax.set_ylabel('LibriParty F1 Drop')
ax.set_title('Catastrophic Forgetting')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)

plt.suptitle('All Three Fine-tuning Strategies: Gain vs. Forgetting', y=1.02)
plt.tight_layout()
plt.savefig('fig_all_strategies.png', bbox_inches='tight')
plt.show()

print(f'Macro TORGO gain  — lr=1e-4: {np.mean(gains_1e4):+.4f} | lr=5e-5: {np.mean(gains_5e5):+.4f} | Mixed: {np.mean(gains_mix):+.4f}')
print(f'Macro Libri drop  — lr=1e-4: {np.mean(drops_1e4):+.4f} | lr=5e-5: {np.mean(drops_5e5):+.4f} | Mixed: {np.mean(drops_mix):+.4f}')

---
## 9. Gain vs. Forgetting Scatter — Trade-off Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

markers = ['o', 's', '^']
for (label, strat, color), marker in zip(strategies, markers):
    g = [strat['gain'][s] for s in speakers]
    d = [strat['drop'][s] for s in speakers]
    ax.scatter(g, d, s=90, color=color, marker=marker, label=label,
               edgecolors='white', linewidths=0.8, zorder=4)
    # Connect same speaker across strategies with light lines
    for i, spk in enumerate(speakers):
        ax.annotate(spk, (g[i], d[i]),
                    fontsize=7.5, ha='left', va='bottom',
                    xytext=(4, 2), textcoords='offset points', color='gray')

# Quadrant lines
ax.axhline(0.05, color='red', ls='--', lw=1.2, alpha=0.7, label='Forgetting threshold (0.05)')
ax.axvline(0.00, color='gray', ls='-', lw=0.7, alpha=0.5)

# Shade ideal region: high gain, low forgetting
ax.axhspan(-0.02, 0.05, color='#d4edda', alpha=0.25)
ax.text(0.14, 0.01, 'Ideal zone\n(high gain, low forgetting)', fontsize=9,
        color='#4CAF50', alpha=0.8)

ax.set_xlabel('TORGO F1 Gain (fine-tuned − pre-trained)', fontsize=11)
ax.set_ylabel('LibriParty F1 Drop (forgetting)', fontsize=11)
ax.set_title('Adaptation–Forgetting Trade-off per Speaker & Strategy')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_scatter_tradeoff.png', bbox_inches='tight')
plt.show()

---
## 10. Summary Table — All Metrics

In [ ]:
rows = []
for s in speakers + ['Macro']:
    if s == 'Macro':
        row = {'Speaker': 'Macro avg',
               'Pre TORGO F1': np.mean(list(pre_f1.values()))}
        for label, strat, _ in strategies:
            row[f'{label} TORGO F1'] = np.mean(list(strat['torgo_f1'].values()))
            row[f'{label} Gain']     = np.mean(list(strat['gain'].values()))
            row[f'{label} Libri F1'] = np.mean(list(strat['libri_f1'].values()))
            row[f'{label} Drop']     = np.mean(list(strat['drop'].values()))
    else:
        row = {'Speaker': s, 'Pre TORGO F1': pre_f1[s]}
        for label, strat, _ in strategies:
            row[f'{label} TORGO F1'] = strat['torgo_f1'][s]
            row[f'{label} Gain']     = strat['gain'][s]
            row[f'{label} Libri F1'] = strat['libri_f1'][s]
            row[f'{label} Drop']     = strat['drop'][s]
    rows.append(row)

df = pd.DataFrame(rows).set_index('Speaker')

# Format floats
float_cols = [c for c in df.columns if df[c].dtype == float]
df_fmt = df[float_cols].applymap(lambda x: f'{x:+.4f}' if 'Gain' in '' else f'{x:.4f}')

# Print grouped
print('=== Full Results Summary ===')
print(f'Pre-trained TinyVAD  — LibriParty F1: {BASELINE_LIBRI:.4f} | TORGO F1: {BASELINE_TORGO:.4f}')
print(f'CRDNN Teacher        — LibriParty F1: {TEACHER_LIBRI:.4f} | TORGO F1: {TEACHER_TORGO:.4f}')
print()

for label, strat, _ in strategies:
    macro_g = np.mean(list(strat['gain'].values()))
    macro_d = np.mean(list(strat['drop'].values()))
    n_forget = sum(1 for s in speakers if strat['drop'][s] > 0.05)
    print(f'{label:20s}  Macro Gain: {macro_g:+.4f}  Macro Drop: {macro_d:.4f}  Forgot: {n_forget}/8')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

strat_labels = ['LOSO\nlr=1e-4', 'LOSO\nlr=5e-5', 'Mixed-data']
all_strats = [loso1e4, loso5e5, mixed]

# ── Gain heatmap ──────────────────────────────────────────────────────────
ax = axes[0]
gain_matrix = np.array([[strat['gain'][s] for s in speakers] for strat in all_strats])
im = ax.imshow(gain_matrix, cmap='RdYlGn', vmin=-0.15, vmax=0.20, aspect='auto')
ax.set_xticks(range(len(speakers))); ax.set_xticklabels(speakers, rotation=30, ha='right')
ax.set_yticks(range(3)); ax.set_yticklabels(strat_labels)
ax.set_title('TORGO F1 Gain')
for i in range(3):
    for j in range(len(speakers)):
        v = gain_matrix[i, j]
        ax.text(j, i, f'{v:+.3f}', ha='center', va='center',
                fontsize=8.5, color='black' if abs(v) < 0.12 else 'white', fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.04)

# ── Drop heatmap ──────────────────────────────────────────────────────────
ax = axes[1]
drop_matrix = np.array([[strat['drop'][s] for s in speakers] for strat in all_strats])
im2 = ax.imshow(drop_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.10, aspect='auto')
ax.set_xticks(range(len(speakers))); ax.set_xticklabels(speakers, rotation=30, ha='right')
ax.set_yticks(range(3)); ax.set_yticklabels(strat_labels)
ax.set_title('LibriParty F1 Drop (Forgetting)')
for i in range(3):
    for j in range(len(speakers)):
        v = drop_matrix[i, j]
        marker = ' !' if v > 0.05 else ''
        ax.text(j, i, f'{v:.3f}{marker}', ha='center', va='center',
                fontsize=8.5, color='black' if v < 0.07 else 'white', fontweight='bold')
plt.colorbar(im2, ax=ax, fraction=0.04)

plt.suptitle('Heatmaps: Gain and Forgetting across Strategies and Speakers', y=1.02)
plt.tight_layout()
plt.savefig('fig_heatmaps.png', bbox_inches='tight')
plt.show()

---
## 11. Macro-level Comparison — Final Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

labels   = ['Pre-trained', 'LOSO lr=1e-4', 'LOSO lr=5e-5', 'Mixed-data', 'Teacher']
colors   = ['#aec6e8', '#4C72B0', '#DD8452', '#55A868', '#9467bd']

# Macro TORGO F1
torgo_vals = [
    BASELINE_TORGO,
    np.mean(list(loso1e4['torgo_f1'].values())),
    np.mean(list(loso5e5['torgo_f1'].values())),
    np.mean(list(mixed['torgo_f1'].values())),
    TEACHER_TORGO,
]
axes[0].bar(labels, torgo_vals, color=colors, edgecolor='white')
for i, v in enumerate(torgo_vals):
    axes[0].text(i, v + 0.003, f'{v:.4f}', ha='center', va='bottom', fontsize=8.5)
axes[0].set_ylim(0.55, 0.90)
axes[0].set_title('Macro TORGO F1 (mean of 8 speakers)')
axes[0].set_ylabel('F1 Score')
axes[0].set_xticklabels(labels, rotation=20, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Macro LibriParty F1
libri_vals = [
    BASELINE_LIBRI,
    np.mean(list(loso1e4['libri_f1'].values())),
    np.mean(list(loso5e5['libri_f1'].values())),
    np.mean(list(mixed['libri_f1'].values())),
    TEACHER_LIBRI,
]
axes[1].bar(labels, libri_vals, color=colors, edgecolor='white')
for i, v in enumerate(libri_vals):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=8.5)
axes[1].set_ylim(0.82, 0.98)
axes[1].set_title('Macro LibriParty F1 (retention)')
axes[1].set_ylabel('F1 Score')
axes[1].set_xticklabels(labels, rotation=20, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# Macro forgetting (only fine-tuned models)
ft_labels = ['LOSO lr=1e-4', 'LOSO lr=5e-5', 'Mixed-data']
ft_colors = ['#4C72B0', '#DD8452', '#55A868']
drop_vals = [
    np.mean(list(loso1e4['drop'].values())),
    np.mean(list(loso5e5['drop'].values())),
    np.mean(list(mixed['drop'].values())),
]
axes[2].bar(ft_labels, drop_vals, color=ft_colors, edgecolor='white')
axes[2].axhline(0.05, color='red', ls='--', lw=1.2, label='Threshold 0.05')
for i, v in enumerate(drop_vals):
    axes[2].text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=8.5)
axes[2].set_ylim(0, 0.09)
axes[2].set_title('Macro LibriParty Drop')
axes[2].set_ylabel('F1 Drop')
axes[2].set_xticklabels(ft_labels, rotation=20, ha='right')
axes[2].grid(axis='y', alpha=0.3)
axes[2].legend()

plt.suptitle('Macro-level Comparison Across All Strategies', y=1.02)
plt.tight_layout()
plt.savefig('fig_macro_comparison.png', bbox_inches='tight')
plt.show()

---
## 12. Key Findings & Conclusions

### 12.1 Knowledge Distillation
- TinyVAD (9,937 params, **11× smaller** than the teacher) reaches **F1 = 0.9223** on LibriParty validation,
  closing 95% of the gap to the teacher (0.9463) while using only 9% of the parameters.
- Training converges rapidly: F1 exceeds 0.90 by **epoch 2**, and most of the gain is captured in the first 5 epochs.
- The remaining gap (0.024 F1) is likely irreducible given the model capacity.

### 12.2 TORGO Adaptation — Three Strategies

| Strategy | Macro TORGO Gain | Macro Libri Drop | Speakers Forgot (>0.05) |
|----------|-----------------|-------------------|------------------------|
| LOSO lr=1e-4 | **+0.0637** | 0.0644 | 7/8 |
| LOSO lr=5e-5 | +0.0527 | 0.0433 | 5/8 |
| **Mixed-data** | +0.0654 | **0.0106** | **0/8** |

**Mixed-data replay is the clear winner**: it achieves the highest TORGO gain (+0.0654) while reducing forgetting by **6× compared to the standard LOSO** (drop 0.011 vs. 0.064).

### 12.3 Per-Speaker Observations
- **M02** is the hardest speaker to adapt: all strategies either show no gain or negative gain.
  This is likely due to M02's specific dysarthric pattern being very different from the other training speakers.
- **F04 and MC01** are the easiest to adapt, with gains exceeding +0.13 in multiple strategies.
- **M01** (severe dysarthria) shows small gains — the limited training data and extreme speech characteristics make it difficult.

### 12.4 Learning Rate Effect
- Lower LR (5e-5) notably helps **M02** specifically (drop from 0.0697 → 0.0022, gain from -0.09 → -0.007)
  but hurts adaptation for most other speakers (M01, M04, M05 gain more than halved).
- Mixed-data replay is a **strictly better strategy** than LR reduction: it achieves both lower forgetting AND higher TORGO gain simultaneously.

### 12.5 Why Mixed-data Works
The model is periodically reminded of LibriParty speech patterns during fine-tuning, preventing the
weight updates from drifting too far from the general VAD distribution. This is an implementation of
**experience replay**, a classic continual learning technique — and the results confirm it is highly
effective even with a very small model (9,937 parameters).

In [ ]:
# ── Radar chart: macro metrics across strategies ───────────────────────────
import matplotlib.pyplot as plt
import numpy as np

categories = ['TORGO\nGain', 'Libri\nRetention', 'No\nForgetting\n(0/8)', 'Gain for\nFemales', 'Gain for\nMales']
N = len(categories)

# Normalise each metric to [0,1] range for radar
def norm(val, lo, hi): return (val - lo) / (hi - lo)

female_spk = ['F01', 'F03', 'F04']
male_spk   = ['M01', 'M02', 'M04', 'M05']

radar_data = {}
for label, strat, color in strategies:
    macro_g    = np.mean(list(strat['gain'].values()))
    macro_ret  = np.mean(list(strat['libri_f1'].values()))
    n_no_forget = sum(1 for s in speakers if strat['drop'][s] <= 0.05)
    f_gain     = np.mean([strat['gain'][s] for s in female_spk])
    m_gain     = np.mean([strat['gain'][s] for s in male_spk])
    radar_data[label] = [
        norm(macro_g,   -0.02, 0.12),
        norm(macro_ret,  0.85, 0.93),
        n_no_forget / 8,
        norm(f_gain,  0.0, 0.15),
        norm(m_gain, -0.05, 0.08),
    ]

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], size=8)
ax.set_ylim(0, 1)

for (label, _, color), (lbl, vals) in zip(strategies, radar_data.items()):
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, color=color, lw=2, label=lbl)
    ax.fill(angles, vals_plot, color=color, alpha=0.1)

ax.set_title('Strategy Profile (normalised scores)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15))
plt.tight_layout()
plt.savefig('fig_radar.png', bbox_inches='tight')
plt.show()

---
## Appendix: Saved Figures

| File | Content |
|------|---------|
| `fig_model_size.png` | Teacher vs. Student parameter count |
| `fig_distill_curves.png` | Distillation loss and F1/P/R curves over 30 epochs |
| `fig_per_speaker_torgo.png` | Per-speaker TORGO F1 gain for all three strategies |
| `fig_forgetting.png` | Per-speaker LibriParty F1 drop for all strategies |
| `fig_lr_comparison.png` | LR=1e-4 vs LR=5e-5 side-by-side |
| `fig_all_strategies.png` | All three strategies compared (gain + drop) |
| `fig_scatter_tradeoff.png` | Scatter: gain vs. forgetting per speaker & strategy |
| `fig_heatmaps.png` | Heatmaps of gain and forgetting |
| `fig_macro_comparison.png` | Macro-level bar charts |
| `fig_radar.png` | Radar chart: overall strategy profile |